# 1. 실험 목적

- 상품 이미지에서 소비기한 날짜 추출에 적합한 OCR 엔진 비교
- 정확도와 CPU 추론 속도를 함께 비교

## 2. 공통 설정

Seed 42, 공통 이미지 목록과 EXIF 방향 보정 함수를 준비합니다. 이미지 처리는 Pillow를 사용합니다.

In [ ]:
from pathlib import Path
import random
import time

from PIL import Image, ImageOps

SEED = 42
random.seed(SEED)

# 프로젝트 루트 또는 notebooks 폴더에서 실행합니다.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "data").is_dir() and (p / "predict.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")
DATA_DIR = PROJECT_ROOT / "data"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
image_paths = sorted(
    (p for p in DATA_DIR.rglob("*")
     if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)


def load_oriented_image(path):
    """EXIF 방향을 보정한 RGB 이미지를 반환합니다. 원본 파일은 변경하지 않습니다."""
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


# 이후 OCR 실행 구간은 time.perf_counter()로 측정합니다.
print(f"전체 이미지: {len(image_paths):,}장")

## 3. 평가용 표본 구성

전체 데이터에서 재현 가능한 방식으로 30장을 선택합니다. 모든 엔진은 동일한 표본과 EXIF 보정 조건을 사용합니다. 정답 라벨은 아직 만들지 않습니다.

In [ ]:
SAMPLE_COUNT = 30
if len(image_paths) < SAMPLE_COUNT:
    raise ValueError(f"이미지가 {SAMPLE_COUNT}장 이상 필요합니다: 현재 {len(image_paths)}장")

# 정렬된 목록에서 독립 RNG로 추출하여 재실행 시에도 같은 표본을 사용합니다.
sample_paths = random.Random(SEED).sample(image_paths, SAMPLE_COUNT)
sample_records = []
for path in sample_paths:
    with Image.open(path) as image:
        width, height = image.size  # EXIF 보정 전 원본 해상도
    sample_records.append({
        "image_id": path.relative_to(DATA_DIR).as_posix(),
        "filename": path.name,
        "original_width": width,
        "original_height": height,
    })

# 정답 라벨 없이 파일명과 원본 해상도만 기록합니다.
sample_records

## 4. OCR 후보

- EasyOCR
- PaddleOCR
- 필요시 Tesseract

## 5. 비교 결과 기록용 빈 표

| engine | image_id | detected_text | detected_date | inference_time_sec | notes |
| --- | --- | --- | --- | --- | --- |

## 6. 실패 유형 기록

- 날짜 미검출
- 날짜 오인식
- 잘못된 숫자를 날짜로 선택
- 회전/반사/과노출
- 작은 글씨
- 기타

## 7. 다음 단계

- OCR 엔진별 동일 표본 실행
- 날짜 인식률 비교
- CPU 속도 비교
- 실패 사례 분석 후 전처리/후처리 설계